In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, coint
import statsmodels.api as sm

# Step 1: Stationarity tests (ADF Test for both series)
def check_stationarity(series, name):
    result = adfuller(series)
    print(f"\n{name} - Augmented Dickey-Fuller Test:")
    print(f"ADF Statistic: {result[0]}")
    print(f"p-value: {result[1]}")
    for key, value in result[4].items():
        print('Critical Value (%s): %.3f' % (key, value))
    if result[1] < 0.05:
        print("=> Stationary")
    else:
        print("=> Not stationary")

# Step 2: Cointegration test
def test_cointegration(df):
    score, pvalue, _ = coint(df['KO_Close'], df['PEP_Close'])
    print(f"\nEngle-Granger Cointegration Test:")
    print(f"t-statistic: {score}")
    print(f"p-value: {pvalue}")
    if pvalue < 0.05:
        print("=> The series are cointegrated")
    else:
        print("=> The series are not cointegrated")

# Step 3: Spread calculation (OLS regression and residuals)
def calculate_spread(df):
    X = df['PEP_Close']
    y = df['KO_Close']
    X = sm.add_constant(X)  # Adds a constant term to the predictor
    model = sm.OLS(y, X)
    results = model.fit()
    print("\nOLS Regression Results:")
    print(results.summary())
    spread = y - results.predict(X)
    df['Spread'] = spread
    print("\nSample of spread values:")
    print(df['Spread'].head())
    return df['Spread']

# Assuming df is your cleaned DataFrame from load_data
if df is not None:

    # 1. Stationarity tests for both series
    check_stationarity(df['KO_Close'], 'KO_Close')
    check_stationarity(df['PEP_Close'], 'PEP_Close')

    # 2. Cointegration test
    test_cointegration(df)

    # 3. Spread calculation
    spread_series = calculate_spread(df)
else:
    print("DataFrame is None; no further analysis performed.")
